# Projeto Final PSI3472 — Planejamento de Voos Nacionais com Algoritmo Genético

**Fluxo de avaliação de um indivíduo**

```
GENOMA ──decodificar──► TAB_AV (voos + aeronaves) ──alocar──► TAB_AP (passageiros) ──► FITNESS
  ▲                                                                                        │
  └──────────────── seleção / cruzamento / mutação atuam só sobre o genoma ◄───────────────┘
```

**Organização do notebook**

| Seção | Conteúdo | Status |
|---|---|---|
| 0 | Dados do enunciado (TAB_TV, TAB_OD) e parâmetros | ✅ |
| 1 | Genoma: estrutura, codificador, decodificador e tabelas (TAB_AV, TAB_PV, TAB_MV) | ✅ |
| 2 | Alocação de passageiros nos voos (TAB_AP) | a implementar |
| 3 | Função de avaliação (fitness) | a implementar |
| 4+ | Algoritmo genético e simulações | a implementar |

In [1]:
import random
import time
from dataclasses import dataclass, field

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

---
## 0. Dados e parâmetros

Parâmetros do enunciado e as hipóteses adotadas pelo grupo onde o enunciado é omisso.

In [2]:
# ---- Parâmetros do enunciado ----
N_AERONAVES        = 400   # frota disponível
CAPACIDADE         = 200   # passageiros por aeronave
PRIMEIRA_DECOLAGEM = 6     # 06:00
ULTIMA_DECOLAGEM   = 21    # 21:00 — última decolagem permitida (chegadas podem ser depois)
ESCALA_MIN_H       = 1     # escala mínima do passageiro: 60 min (usada na Seção 2)

# ---- Hipóteses do grupo (não especificadas no enunciado) ----
TURNAROUND_H = 1           # tempo mínimo da aeronave em solo entre pousar e decolar de novo

# ---- Parâmetros do genoma ----
MAX_TRECHOS = 8            # nº máximo de voos por aeronave no dia (enunciado: referência de 2 a 8)
ESPERA_MAX  = 3            # espera extra máxima (h) sorteada para cada trecho na geração de genomas

### 0.1 Aeroportos e TAB_TV — tempos de voo (min)
Linha = origem, coluna = destino. A diagonal é 0 (não há voo de um aeroporto para ele mesmo).
Os totais de linha/coluna são conferidos contra os do PDF para evitar erro de digitação.

In [3]:
AEROPORTOS = ["MA", "BE", "FO", "NA", "RE", "SA", "BR", "BH", "CI", "RJ", "SP", "CR", "FL", "PA"]
NOMES = {
    "MA": "Manaus", "BE": "Belém", "FO": "Fortaleza", "NA": "Natal", "RE": "Recife",
    "SA": "Salvador", "BR": "Brasília", "BH": "Belo Horizonte", "CI": "Cuiabá",
    "RJ": "Rio de Janeiro", "SP": "São Paulo", "CR": "Curitiba", "FL": "Florianópolis",
    "PA": "Porto Alegre",
}

_TV = [
#     MA   BE   FO   NA   RE   SA   BR   BH   CI   RJ   SP   CR   FL   PA
    [  0, 120, 180, 180, 180, 180, 180, 240, 180, 300, 300, 300, 360, 360],  # MA
    [120,   0, 120, 120, 120, 180, 180, 240, 180, 240, 240, 300, 300, 360],  # BE
    [180, 120,   0,  60,  60, 120, 180, 180, 240, 240, 240, 300, 300, 360],  # FO
    [180, 120,  60,   0,  60, 120, 180, 180, 240, 240, 240, 300, 300, 360],  # NA
    [180, 120,  60,  60,   0, 120, 180, 180, 240, 180, 180, 240, 240, 300],  # RE
    [180, 180, 120, 120, 120,   0, 120, 120, 180, 120, 120, 180, 180, 240],  # SA
    [180, 180, 180, 180, 180, 120,   0,  60,  60, 120, 120, 180, 180, 240],  # BR
    [240, 240, 180, 180, 180, 120,  60,   0, 120,  60,  60, 120, 120, 180],  # BH
    [180, 180, 240, 240, 240, 180,  60, 120,   0, 180, 180, 180, 180, 240],  # CI
    [300, 240, 240, 240, 180, 120, 120,  60, 180,   0,  60, 120, 120, 180],  # RJ
    [300, 240, 240, 240, 180, 120, 120,  60, 180,  60,   0,  60,  60, 120],  # SP
    [300, 300, 300, 300, 240, 180, 180, 120, 180, 120,  60,   0,  60,  60],  # CR
    [360, 300, 300, 300, 240, 180, 180, 120, 180, 120,  60,  60,   0,  60],  # FL
    [360, 360, 360, 360, 300, 240, 240, 180, 240, 180, 120,  60,  60,   0],  # PA
]
TAB_TV = pd.DataFrame(_TV, index=AEROPORTOS, columns=AEROPORTOS)

# Conferência com os totais do PDF
_TOTAIS_TV = [3060, 2700, 2580, 2580, 2280, 1980, 1980, 1860, 2400, 2160, 1980, 2400, 2460, 3060]
assert TAB_TV.sum(axis=1).tolist() == _TOTAIS_TV
assert TAB_TV.sum(axis=0).tolist() == _TOTAIS_TV
# Obs.: o PDF mostra total geral 34480, mas a soma dos próprios totais de linha dá 33480 (erro no PDF).
assert TAB_TV.values.sum() == 33480
assert (TAB_TV.values % 60 == 0).all(), "todos os tempos devem ser múltiplos de 60 min"

# Acesso rápido em HORAS: TV_H[origem][destino]
TV_H = {o: {d: int(TAB_TV.at[o, d]) // 60 for d in AEROPORTOS if d != o} for o in AEROPORTOS}

TAB_TV

,MA,BE,FO,NA,RE,SA,BR,BH,CI,RJ,SP,CR,FL,PA
MA,0,120,180,180,180,180,180,240,180,300,300,300,360,360
BE,120,0,120,120,120,180,180,240,180,240,240,300,300,360
FO,180,120,0,60,60,120,180,180,240,240,240,300,300,360
NA,180,120,60,0,60,120,180,180,240,240,240,300,300,360
RE,180,120,60,60,0,120,180,180,240,180,180,240,240,300
SA,180,180,120,120,120,0,120,120,180,120,120,180,180,240
BR,180,180,180,180,180,120,0,60,60,120,120,180,180,240
BH,240,240,180,180,180,120,60,0,120,60,60,120,120,180
CI,180,180,240,240,240,180,60,120,0,180,180,180,180,240
RJ,300,240,240,240,180,120,120,60,180,0,60,120,120,180


### 0.2 TAB_OD — demanda diária de passageiros (origem → destino)
Cada passageiro é contado uma vez pela viagem completa (com ou sem escala).

In [4]:
_OD = [
#      MA    BE    FO    NA    RE    SA    BR    BH    CI    RJ    SP    CR    FL    PA
    [   0,  800,  900,  600,  900,  900, 1600,  800,  500, 1700, 2200,  800,  400,  600],  # MA
    [ 900,    0, 1100,  800,  700, 1100, 1400,  900,  300, 1800, 1800,  800,  400,  500],  # BE
    [ 600,  900,    0, 1200, 1200, 1200, 1700, 1000,  400, 2300, 1600,  900,  600,  800],  # FO
    [ 400,  700, 1000,    0,  900, 1200, 1400,  600,  600, 1400, 1600,  900,  600,  700],  # NA
    [ 600,  800, 1500, 1100,    0, 1400, 1900,  700,  900, 2600, 2400, 1100,  900,  900],  # RE
    [ 400,  500, 1600, 1500, 1400,    0, 2100, 1100, 1100, 2400, 2500, 1100, 1100, 1100],  # SA
    [ 900, 1100, 1800, 1400, 1700, 2100,    0, 1500, 1700, 3600, 7800, 1300, 1200, 1400],  # BR
    [ 400,  900, 1100,  600, 1200,  800, 2200,    0,  800, 1400, 1800,  700,  500,  600],  # BH
    [ 500,  500,  900,  800,  900,  600, 1000,  700,    0, 1000, 1400,  500,  400,  500],  # CI
    [1200,  900, 2000, 1500, 2100, 2400, 3700, 1900, 1800,    0, 8100, 1700, 1500, 1700],  # RJ
    [1500, 1600, 3000, 1800, 3100, 2600, 5900,  600, 1700, 7000,    0, 3000, 2100, 2200],  # SP
    [ 300,  800, 1000,  700,  800, 1200, 1400,  900,  600, 1600, 1600,    0,  700, 1100],  # CR
    [ 300,  700,  900, 1400,  900, 1300, 1600, 1100,  600, 2000, 1800, 1000,    0,  700],  # FL
    [ 300,  600,  800, 1100,  500, 1100, 1500,  900,  400, 1900, 1500,  700,  600,    0],  # PA
]
TAB_OD = pd.DataFrame(_OD, index=AEROPORTOS, columns=AEROPORTOS)

# Conferência com os totais do PDF
assert TAB_OD.sum(axis=1).tolist() == [12700, 12500, 14400, 12000, 16800, 17900, 27500,
                                       13000, 9700, 30500, 36100, 12700, 14300, 11900]
assert TAB_OD.sum(axis=0).tolist() == [8300, 10800, 17600, 14500, 16300, 17900, 27400,
                                       12700, 11400, 30700, 36100, 14500, 11000, 12800]
assert TAB_OD.values.sum() == 242_000

# Acesso rápido: DEMANDA[(origem, destino)]
TRECHOS = [(o, d) for o in AEROPORTOS for d in AEROPORTOS if o != d]   # 182 pares O-D
DEMANDA = {(o, d): int(TAB_OD.at[o, d]) for o, d in TRECHOS}

# Referência: nº mínimo de voos diretos para levar toda a demanda sem escalas
voos_min = sum(-(-n // CAPACIDADE) for n in DEMANDA.values())
print(f"Demanda total: {sum(DEMANDA.values()):,} passageiros/dia em {len(TRECHOS)} pares O-D")
print(f"Voos diretos mínimos (ceil(demanda/200) por par): {voos_min}  "
      f"| capacidade de referência: {N_AERONAVES} aeronaves x 5 voos = {N_AERONAVES * 5}")

TAB_OD

Demanda total: 242,000 passageiros/dia em 182 pares O-D
Voos diretos mínimos (ceil(demanda/200) por par): 1255  | capacidade de referência: 400 aeronaves x 5 voos = 2000


,MA,BE,FO,NA,RE,SA,BR,BH,CI,RJ,SP,CR,FL,PA
MA,0,800,900,600,900,900,1600,800,500,1700,2200,800,400,600
BE,900,0,1100,800,700,1100,1400,900,300,1800,1800,800,400,500
FO,600,900,0,1200,1200,1200,1700,1000,400,2300,1600,900,600,800
NA,400,700,1000,0,900,1200,1400,600,600,1400,1600,900,600,700
RE,600,800,1500,1100,0,1400,1900,700,900,2600,2400,1100,900,900
SA,400,500,1600,1500,1400,0,2100,1100,1100,2400,2500,1100,1100,1100
BR,900,1100,1800,1400,1700,2100,0,1500,1700,3600,7800,1300,1200,1400
BH,400,900,1100,600,1200,800,2200,0,800,1400,1800,700,500,600
CI,500,500,900,800,900,600,1000,700,0,1000,1400,500,400,500
RJ,1200,900,2000,1500,2100,2400,3700,1900,1800,0,8100,1700,1500,1700


---
## 1. Genoma — codificação, decodificação e tabelas

### Estrutura do genoma

```
genoma = [ gene_A001, gene_A002, ..., gene_A400 ]          (um gene por aeronave)

gene   = base | (destino₁, espera₁) | (destino₂, espera₂) | ... | (destino₈, espera₈)
```

- **base**: aeroporto onde a aeronave amanhece (primeiro voo parte daí, a partir das 06:00).
- **destino**: próximo aeroporto. O valor especial `FIM` encerra o dia daquela aeronave.
- **espera**: horas extras em solo antes de decolar, além do turnaround.

**Decisões de projeto**

- A **origem** de cada voo não está no genoma: é sempre o destino do voo anterior ⇒ é impossível uma aeronave "se teletransportar", mesmo após cruzamento/mutação.
- Os **horários** também não estão no genoma: são calculados pelo decodificador
  (`partida = disponível + espera`, `chegada = partida + TV`, `disponível = chegada + turnaround`) ⇒ nunca ficam inconsistentes.
- Trechos que partiriam depois das 21:00 são **inativos** (ignorados pelo decodificador), como "DNA não-expresso". Podem voltar a ser ativos após uma mutação que antecipe horários.
- **Passageiros não estão no genoma**: o nº de passageiros de cada voo é consequência da demanda e é calculado pelo alocador (Seção 2).

### 1.1 O gene da aeronave

In [5]:
FIM = None   # destino especial: a aeronave encerra o dia


@dataclass
class GeneAeronave:
    """Gene de uma aeronave: onde ela amanhece e a sequência de (destino, espera)."""
    base: str
    trechos: list   # list[tuple[str | None, int]] com MAX_TRECHOS elementos

    def copia(self):
        return GeneAeronave(self.base, list(self.trechos))

    def __str__(self):
        partes = [self.base]
        for destino, espera in self.trechos:
            if destino is FIM:
                partes.append("FIM")
                break
            partes.append(f"{destino}+{espera}")
        return " | ".join(partes)


def mostrar_genoma(genoma, n=5):
    for i, gene in enumerate(genoma[:n], start=1):
        print(f"A{i:03d}: {gene}")
    if len(genoma) > n:
        print(f"... ({len(genoma) - n} genes omitidos)")

### 1.2 Fenótipo: `Voo`, `Aeronave` e `Plano`

O fenótipo é o que o genoma "produz". Cada indivíduo tem o seu próprio `Plano`.
`Voo.ocupacao` fica vazio aqui e será preenchido pelo alocador na Seção 2, no formato
`{(origem_final, destino_final): n_passageiros}`.

In [6]:
def fmt_hora(h):
    """Formata hora inteira; chegadas após meia-noite aparecem como (+1d)."""
    return f"{h:02d}:00" if h < 24 else f"{h - 24:02d}:00(+1d)"


@dataclass(eq=False)
class Voo:
    id: str            # "V0001"
    aeronave: str      # "A001"
    ordem: int         # posição do voo na sequência da aeronave (1, 2, ...)
    origem: str
    destino: str
    partida: int       # hora cheia
    chegada: int       # hora cheia (pode passar de 24)
    ocupacao: dict = field(default_factory=dict)   # {(orig_final, dest_final): n} — Seção 2

    @property
    def total(self):
        return sum(self.ocupacao.values())

    @property
    def livres(self):
        return CAPACIDADE - self.total

    @property
    def duracao_min(self):
        return (self.chegada - self.partida) * 60

    @property
    def rotulo(self):
        """Rótulo no estilo do PDF (ex.: A1V2 = aeronave 1, 2º voo)."""
        return f"A{int(self.aeronave[1:])}V{self.ordem}"

    def __repr__(self):
        return (f"{self.id}[{self.aeronave} {self.origem}→{self.destino} "
                f"{fmt_hora(self.partida)}-{fmt_hora(self.chegada)} {self.total}pax]")


@dataclass(eq=False)
class Aeronave:
    id: str
    base: str
    capacidade: int = CAPACIDADE
    voos: list = field(default_factory=list)

    @property
    def usada(self):
        return bool(self.voos)

    @property
    def horas_voo(self):
        return sum(v.chegada - v.partida for v in self.voos)


@dataclass
class Plano:
    aeronaves: list          # list[Aeronave]
    voos: list               # list[Voo]  (TAB_AV)
    voos_por_trecho: dict    # {(origem, destino): [Voo, ...] ordenados por partida} — índice p/ Seção 2

### 1.3 Decodificador: genoma → `Plano` (TAB_AV)

In [7]:
def decodificar(genoma):
    """Transforma o genoma em voos concretos (com horários) atribuídos às aeronaves."""
    aeronaves, voos, voos_por_trecho = [], [], {}

    for i, gene in enumerate(genoma, start=1):
        av = Aeronave(f"A{i:03d}", gene.base)
        local, disponivel = gene.base, PRIMEIRA_DECOLAGEM

        for destino, espera in gene.trechos:
            if destino is FIM:
                break
            if destino == local:
                raise ValueError(f"{av.id}: trecho inválido {local}→{destino} (origem = destino)")
            partida = disponivel + espera
            if partida > ULTIMA_DECOLAGEM:          # trechos restantes ficam inativos
                break
            chegada = partida + TV_H[local][destino]

            voo = Voo(f"V{len(voos) + 1:04d}", av.id, len(av.voos) + 1,
                      local, destino, partida, chegada)
            av.voos.append(voo)
            voos.append(voo)
            voos_por_trecho.setdefault((local, destino), []).append(voo)

            local, disponivel = destino, chegada + TURNAROUND_H

        aeronaves.append(av)

    for lista in voos_por_trecho.values():
        lista.sort(key=lambda v: v.partida)

    return Plano(aeronaves, voos, voos_por_trecho)

### 1.4 Codificador: `Plano` → genoma

Inverso do decodificador. Serve para transformar um plano montado à mão (ou por uma heurística)
em indivíduo do AG, e para normalizar genomas (trechos inativos viram `FIM`).
Também verifica a consistência do plano: continuidade da aeronave, turnaround e tempos de voo.

In [8]:
def codificar(plano):
    """Reconstrói o genoma a partir da sequência de voos de cada aeronave."""
    genoma = []
    for av in plano.aeronaves:
        local, disponivel = av.base, PRIMEIRA_DECOLAGEM
        trechos = []
        for v in av.voos:
            if v.origem != local:
                raise ValueError(f"{v.id}: aeronave {av.id} está em {local}, não em {v.origem}")
            espera = v.partida - disponivel
            if espera < 0:
                raise ValueError(f"{v.id}: parte às {fmt_hora(v.partida)}, antes da aeronave "
                                 f"estar disponível ({fmt_hora(disponivel)})")
            if v.chegada - v.partida != TV_H[v.origem][v.destino]:
                raise ValueError(f"{v.id}: duração não confere com a TAB_TV")
            trechos.append((v.destino, espera))
            local, disponivel = v.destino, v.chegada + TURNAROUND_H

        if len(trechos) > MAX_TRECHOS:
            raise ValueError(f"{av.id}: {len(trechos)} voos > MAX_TRECHOS={MAX_TRECHOS}")
        trechos += [(FIM, 0)] * (MAX_TRECHOS - len(trechos))
        genoma.append(GeneAeronave(av.base, trechos))
    return genoma

### 1.5 Geração de genomas (população inicial)

Duas formas de criar indivíduos:

- **Aleatório**: base e destinos uniformes, espera uniforme em `0..ESPERA_MAX`. Garante diversidade.
- **Semeado pela demanda**: base sorteada proporcionalmente à demanda de saída de cada aeroporto;
  cada destino sorteado proporcionalmente à demanda `TAB_OD[local][destino]`; esperas curtas são mais prováveis.
  Começa o AG mais perto de uma boa solução.

In [9]:
_DESTINOS = {o: [d for d in AEROPORTOS if d != o] for o in AEROPORTOS}
_PESOS_DEMANDA = {o: [DEMANDA[(o, d)] for d in _DESTINOS[o]] for o in AEROPORTOS}
_PESOS_BASE = TAB_OD.sum(axis=1).tolist()                       # demanda de saída
_ESPERAS = list(range(ESPERA_MAX + 1))
_PESOS_ESPERA = [0.5 ** k for k in _ESPERAS]                     # 0h mais provável, depois 1h, ...


def gene_aleatorio(rng):
    base = rng.choice(AEROPORTOS)
    trechos, local = [], base
    for _ in range(MAX_TRECHOS):
        destino = rng.choice(_DESTINOS[local])
        trechos.append((destino, rng.randint(0, ESPERA_MAX)))
        local = destino
    return GeneAeronave(base, trechos)


def gene_por_demanda(rng):
    base = rng.choices(AEROPORTOS, weights=_PESOS_BASE)[0]
    trechos, local = [], base
    for _ in range(MAX_TRECHOS):
        destino = rng.choices(_DESTINOS[local], weights=_PESOS_DEMANDA[local])[0]
        espera = rng.choices(_ESPERAS, weights=_PESOS_ESPERA)[0]
        trechos.append((destino, espera))
        local = destino
    return GeneAeronave(base, trechos)


def genoma_aleatorio(rng, n=N_AERONAVES):
    return [gene_aleatorio(rng) for _ in range(n)]


def genoma_por_demanda(rng, n=N_AERONAVES):
    return [gene_por_demanda(rng) for _ in range(n)]


def populacao_inicial(tamanho, frac_demanda=0.5, seed=None):
    """Cria `tamanho` genomas: uma fração semeada pela demanda e o restante aleatório."""
    rng = random.Random(seed)
    n_dem = round(tamanho * frac_demanda)
    return ([genoma_por_demanda(rng) for _ in range(n_dem)] +
            [genoma_aleatorio(rng) for _ in range(tamanho - n_dem)])

### 1.6 Tabelas geradas a partir do `Plano`

| Tabela | Função | Conteúdo |
|---|---|---|
| TAB_AV | `tabela_av` | um voo por linha: aeronave, trecho, horários, nº de passageiros |
| TAB_PV | `plano_diario_por_aeronave`, `plano_diario_por_voo`, `tabela_pv` | plano diário nas duas formas do PDF |
| TAB_MV | `tabela_mv` | mapa espaço × tempo: 182 trechos × 16 horários de partida |

Além disso, `validar_plano` confere as restrições de forma independente do decodificador
e `resumo_plano` resume os números do plano.

In [10]:
def tabela_av(plano):
    """TAB_AV — Alocação de Voos: vooVVVV: [aeronave] [(Orig)(Dest)] [(Part)(Cheg)] [#passag]."""
    return pd.DataFrame([{
        "voo": v.id, "aeronave": v.aeronave, "origem": v.origem, "destino": v.destino,
        "partida": fmt_hora(v.partida), "chegada": fmt_hora(v.chegada),
        "duracao_min": v.duracao_min, "n_passag": v.total,
    } for v in plano.voos]).set_index("voo")


def plano_diario_por_aeronave(plano):
    """TAB_PV (forma 1): { aeronave: [voos...] }."""
    return {av.id: [v.id for v in av.voos] for av in plano.aeronaves}


def plano_diario_por_voo(plano):
    """TAB_PV (forma 2): lista de todos os voos, em ordem de partida."""
    return [v.id for v in sorted(plano.voos, key=lambda v: (v.partida, v.id))]


def tabela_pv(plano):
    """TAB_PV em formato de tabela, uma linha por aeronave."""
    return pd.DataFrame([{
        "aeronave": av.id, "base": av.base, "n_voos": len(av.voos), "horas_voo": av.horas_voo,
        "termina_em": av.voos[-1].destino if av.voos else av.base,
        "sequencia": " → ".join(f"{v.id} {v.origem}-{v.destino} {v.partida:02d}h" for v in av.voos),
    } for av in plano.aeronaves]).set_index("aeronave")


_HORAS = list(range(PRIMEIRA_DECOLAGEM, ULTIMA_DECOLAGEM + 1))      # 16 horários de partida
_LINHA_MV = {t: i for i, t in enumerate(TRECHOS)}


def tabela_mv(plano, modo="rotulo"):
    """TAB_MV — Mapa de Voos (182 trechos x 16 horários de partida).
    modo="rotulo": célula com os rótulos dos voos (ex.: A7V2); modo="contagem": nº de voos."""
    if modo == "contagem":
        dados = np.zeros((len(TRECHOS), len(_HORAS)), dtype=int)
    else:
        dados = np.full((len(TRECHOS), len(_HORAS)), "", dtype=object)
    for v in plano.voos:
        i, j = _LINHA_MV[(v.origem, v.destino)], v.partida - PRIMEIRA_DECOLAGEM
        if modo == "contagem":
            dados[i, j] += 1
        else:
            dados[i, j] = f"{dados[i, j]},{v.rotulo}" if dados[i, j] else v.rotulo
    idx = pd.MultiIndex.from_tuples(TRECHOS, names=["origem", "destino"])
    return pd.DataFrame(dados, index=idx, columns=[fmt_hora(h) for h in _HORAS])


def validar_plano(plano):
    """Confere as restrições independentemente do decodificador. Retorna a lista de erros."""
    erros = []
    for av in plano.aeronaves:
        local, disponivel = av.base, PRIMEIRA_DECOLAGEM
        for v in av.voos:
            if v.origem != local:
                erros.append(f"{v.id}: aeronave {av.id} em {local}, voo sai de {v.origem}")
            if v.partida < disponivel:
                erros.append(f"{v.id}: parte antes do turnaround terminar")
            if not PRIMEIRA_DECOLAGEM <= v.partida <= ULTIMA_DECOLAGEM:
                erros.append(f"{v.id}: decolagem fora da janela 06h-21h")
            if v.chegada - v.partida != TV_H[v.origem][v.destino]:
                erros.append(f"{v.id}: duração diferente da TAB_TV")
            if v.total > CAPACIDADE:
                erros.append(f"{v.id}: {v.total} passageiros > capacidade")
            local, disponivel = v.destino, v.chegada + TURNAROUND_H
    usadas = sum(av.usada for av in plano.aeronaves)
    if usadas > N_AERONAVES:
        erros.append(f"{usadas} aeronaves usadas > frota de {N_AERONAVES}")
    return erros


def resumo_plano(plano):
    usadas = [av for av in plano.aeronaves if av.usada]
    n = len(plano.voos)
    # limite superior de passageiros que poderiam voar SEM escala neste plano
    direto_max = sum(min(dem, CAPACIDADE * len(plano.voos_por_trecho.get(od, [])))
                     for od, dem in DEMANDA.items())
    return pd.Series({
        "voos": n,
        "aeronaves usadas": len(usadas),
        "voos por aeronave (média)": round(n / max(len(usadas), 1), 2),
        "horas de voo (total)": sum(av.horas_voo for av in plano.aeronaves),
        "assentos ofertados": n * CAPACIDADE,
        "pares O-D com voo direto": f"{len(plano.voos_por_trecho)} / {len(TRECHOS)}",
        "máx. passageiros sem escala": direto_max,
    })

### 1.7 Demonstração

**(a) Exemplo trabalhado** — `SP | BR+0 | MA+0 | BR+1 | SP+0 | FIM`

A espera de cada trecho atrasa a decolagem **daquele** trecho: `BR+1` significa "espere 1h extra em MA e depois voe para BR".

| Voo | Trecho | Disponível | + espera | Partida | + TV | Chegada |
|---|---|---|---|---|---|---|
| 1 | SP→BR | 06h | 0 | 06h | 2h | 08h |
| 2 | BR→MA | 08h + 1h turnaround = 09h | 0 | 09h | 3h | 12h |
| 3 | MA→BR | 12h + 1h = 13h | **1** | 14h | 3h | 17h |
| 4 | BR→SP | 17h + 1h = 18h | 0 | 18h | 2h | 20h |

In [11]:
gene_exemplo = GeneAeronave("SP", [("BR", 0), ("MA", 0), ("BR", 1), ("SP", 0)]
                                  + [(FIM, 0)] * (MAX_TRECHOS - 4))
print("Gene:", gene_exemplo)

plano_ex = decodificar([gene_exemplo])
assert [(v.origem, v.destino, v.partida, v.chegada) for v in plano_ex.voos] == [
    ("SP", "BR", 6, 8), ("BR", "MA", 9, 12), ("MA", "BR", 14, 17), ("BR", "SP", 18, 20)]
assert validar_plano(plano_ex) == []

tabela_av(plano_ex)

Gene: SP | BR+0 | MA+0 | BR+1 | SP+0 | FIM


,aeronave,origem,destino,partida,chegada,duracao_min,n_passag
voo,,,,,,,
V0001,A001,SP,BR,06:00,08:00,120,0
V0002,A001,BR,MA,09:00,12:00,180,0
V0003,A001,MA,BR,14:00,17:00,180,0
V0004,A001,BR,SP,18:00,20:00,120,0


**(b) Trechos inativos e normalização** — uma aeronave com 8 trechos que só consegue fazer 3 voos
antes das 21h. O codificador devolve o genoma normalizado (trechos inativos → `FIM`).

In [12]:
gene_longo = GeneAeronave("PA", [("MA", 0), ("PA", 0), ("MA", 0), ("PA", 0),
                                 ("MA", 0), ("PA", 0), ("MA", 0), ("PA", 0)])
plano_longo = decodificar([gene_longo])
print("Gene original:   ", gene_longo)
print("Gene normalizado:", codificar(plano_longo)[0])
print("Voos gerados:    ", plano_longo.voos)

Gene original:    PA | MA+0 | PA+0 | MA+0 | PA+0 | MA+0 | PA+0 | MA+0 | PA+0
Gene normalizado: PA | MA+0 | PA+0 | MA+0 | FIM
Voos gerados:     [V0001[A001 PA→MA 06:00-12:00 0pax], V0002[A001 MA→PA 13:00-19:00 0pax], V0003[A001 PA→MA 20:00-02:00(+1d) 0pax]]


**(c) Erros detectados** — o decodificador rejeita trecho com origem = destino, e o codificador
rejeita um plano em que a aeronave se "teletransporta".

In [13]:
try:
    decodificar([GeneAeronave("SP", [("SP", 0)] + [(FIM, 0)] * (MAX_TRECHOS - 1))])
except ValueError as e:
    print("decodificar:", e)

plano_ruim = decodificar([gene_exemplo])
plano_ruim.aeronaves[0].voos[1].origem = "RJ"        # adultera: o 2º voo sairia do RJ
try:
    codificar(plano_ruim)
except ValueError as e:
    print("codificar:  ", e)
print("validar_plano:", validar_plano(plano_ruim))

decodificar: A001: trecho inválido SP→SP (origem = destino)
codificar:   V0002: aeronave A001 está em BR, não em RJ
validar_plano: ['V0002: aeronave A001 em BR, voo sai de RJ', 'V0002: duração diferente da TAB_TV']


**(d) População inicial** — 10 indivíduos (5 semeados pela demanda + 5 aleatórios),
decodificados e validados. Também conferimos que `decodificar(codificar(plano))` reproduz o mesmo plano.

In [14]:
populacao = populacao_inicial(10, frac_demanda=0.5, seed=42)

t0 = time.perf_counter()
planos = [decodificar(g) for g in populacao]
t_dec = (time.perf_counter() - t0) / len(populacao)

for p in planos:
    assert validar_plano(p) == []
    assert tabela_av(decodificar(codificar(p))).equals(tabela_av(p))   # ida e volta

print(f"Decodificação: {t_dec * 1000:.1f} ms por indivíduo  |  todos os planos válidos ✓")

resumos = pd.DataFrame([resumo_plano(p) for p in planos],
                       index=[f"ind{i:02d}" for i in range(len(planos))])
resumos.insert(0, "tipo", ["demanda"] * 5 + ["aleatório"] * 5)
resumos

Decodificação: 1.9 ms por indivíduo  |  todos os planos válidos ✓

,tipo,voos,aeronaves usadas,voos por aeronave (média),horas de voo (total),assentos ofertados,pares O-D com voo direto,máx. passageiros sem escala
ind00,demanda,1563,400,3.91,4190,312600,182 / 182,226700
ind01,demanda,1553,400,3.88,4267,310600,180 / 182,228200
ind02,demanda,1564,400,3.91,4165,312800,181 / 182,225900
ind03,demanda,1581,400,3.95,4259,316200,181 / 182,229200
ind04,demanda,1577,400,3.94,4220,315400,179 / 182,226000
ind05,aleatório,1204,400,3.01,3764,240800,182 / 182,171600
ind06,aleatório,1231,400,3.08,3734,246200,182 / 182,168200
ind07,aleatório,1229,400,3.07,3684,245800,182 / 182,178100
ind08,aleatório,1244,400,3.11,3756,248800,182 / 182,175400
ind09,aleatório,1232,400,3.08,3741,246400,182 / 182,171000


**(e) Tabelas de um indivíduo** (o 1º da população, semeado pela demanda).
O `n_passag` está zerado porque a alocação de passageiros é a Seção 2.

In [15]:
genoma0, plano0 = populacao[0], planos[0]
print("Genoma (primeiras aeronaves):")
mostrar_genoma(genoma0)
print()
print("TAB_AV (primeiros voos):")
tabela_av(plano0).head(12)

Genoma (primeiras aeronaves):
A001: RJ | MA+0 | RE+1 | RJ+2 | FO+0 | MA+0 | BH+0 | NA+1 | CI+0
A002: RJ | SP+0 | CR+1 | SA+0 | PA+0 | FO+0 | CR+1 | SP+1 | BR+3
A003: BR | RJ+2 | SP+2 | CI+1 | MA+0 | SA+0 | RE+0 | SA+1 | BR+0
A004: NA | SA+3 | RJ+1 | NA+1 | FO+0 | PA+1 | BH+1 | SP+1 | RE+0
A005: SA | RE+0 | FL+2 | SA+1 | BR+2 | RJ+0 | RE+1 | SA+1 | FL+0
... (395 genes omitidos)

TAB_AV (primeiros voos):


,aeronave,origem,destino,partida,chegada,duracao_min,n_passag
voo,,,,,,,
V0001,A001,RJ,MA,06:00,11:00,300,0
V0002,A001,MA,RE,13:00,16:00,180,0
V0003,A001,RE,RJ,19:00,22:00,180,0
V0004,A002,RJ,SP,06:00,07:00,60,0
V0005,A002,SP,CR,09:00,10:00,60,0
V0006,A002,CR,SA,11:00,14:00,180,0
V0007,A002,SA,PA,15:00,19:00,240,0
V0008,A002,PA,FO,20:00,02:00(+1d),360,0
V0009,A003,BR,RJ,08:00,10:00,120,0


In [16]:
print("TAB_PV — por aeronave (formato do PDF):")
for av_id, voos_av in list(plano_diario_por_aeronave(plano0).items())[:3]:
    print(f"  {av_id}: {voos_av}")
print("TAB_PV — por voo (em ordem de partida, primeiros 15):")
print(" ", plano_diario_por_voo(plano0)[:15], "...")
tabela_pv(plano0).head(10)

TAB_PV — por aeronave (formato do PDF):
  A001: ['V0001', 'V0002', 'V0003']
  A002: ['V0004', 'V0005', 'V0006', 'V0007', 'V0008']
  A003: ['V0009', 'V0010', 'V0011', 'V0012']
TAB_PV — por voo (em ordem de partida, primeiros 15):
  ['V0001', 'V0004', 'V0016', 'V0019', 'V0022', 'V0032', 'V0036', 'V0040', 'V0044', 'V0051', 'V0056', 'V0059', 'V0074', 'V0077', 'V0094'] ...


,base,n_voos,horas_voo,termina_em,sequencia
aeronave,,,,,
A001,RJ,3,11,RJ,V0001 RJ-MA 06h → V0002 MA-RE 13h → V0003 RE-R...
A002,RJ,5,15,FO,V0004 RJ-SP 06h → V0005 SP-CR 09h → V0006 CR-S...
A003,BR,4,9,MA,V0009 BR-RJ 08h → V0010 RJ-SP 13h → V0011 SP-C...
A004,NA,3,8,NA,V0013 NA-SA 09h → V0014 SA-RJ 13h → V0015 RJ-N...
A005,SA,3,9,SA,V0016 SA-RE 06h → V0017 RE-FL 11h → V0018 FL-S...
A006,RE,3,12,NA,V0019 RE-PA 06h → V0020 PA-FO 12h → V0021 FO-N...
A007,RJ,4,12,PA,V0022 RJ-CI 06h → V0023 CI-RJ 10h → V0024 RJ-B...
A008,FO,3,10,MA,V0026 FO-SP 07h → V0027 SP-RJ 12h → V0028 RJ-M...
A009,BE,3,12,CI,V0029 BE-SP 07h → V0030 SP-FO 12h → V0031 FO-C...


In [17]:
print("TAB_MV — trechos saindo de Manaus (rótulo AnVk = aeronave n, k-ésimo voo):")
tabela_mv(plano0).loc["MA"]

TAB_MV — trechos saindo de Manaus (rótulo AnVk = aeronave n, k-ésimo voo):


,06:00,07:00,08:00,09:00,10:00,11:00,12:00,13:00,14:00,15:00,16:00,17:00,18:00,19:00,20:00,21:00
destino,,,,,,,,,,,,,,,,
BE,,,,,,,,,,,A398V3,,,A337V3,,
FO,A316V1,,,,,A32V2,A320V2,,,A103V2,A132V3,,,A36V3,,
NA,"A180V1,A297V1",,,,,A70V2,A399V2,A185V2,,,,A397V3,,,,
RE,,A216V1,,,,,,A1V2,,,,,,,,
SA,,,,A352V1,,,,,A159V2,,,,,,,
BR,,A30V1,,,A346V2,A385V2,"A85V2,A392V2",A274V2,,A24V3,,A395V4,,"A153V3,A355V3",,
BH,,,,,"A138V2,A300V2",,,,,,,,,A299V4,,
CI,,,,,A278V2,,,,,,A259V4,,,,,
RJ,A109V1,,,,A247V2,A18V2,,,A214V3,,"A292V3,A386V3",A126V2,,,,


In [18]:
print("Visão geral da TAB_MV: nº de decolagens por aeroporto de origem e horário")
tabela_mv(plano0, modo="contagem").groupby(level="origem", sort=False).sum()

Visão geral da TAB_MV: nº de decolagens por aeroporto de origem e horário


,06:00,07:00,08:00,09:00,10:00,11:00,12:00,13:00,14:00,15:00,16:00,17:00,18:00,19:00,20:00,21:00
origem,,,,,,,,,,,,,,,,
MA,8,3,0,1,5,4,4,3,3,3,7,6,0,5,1,1
BE,8,9,3,4,2,7,4,6,10,7,0,7,2,3,4,4
FO,1,5,6,1,3,4,4,3,7,10,5,9,6,9,11,10
NA,6,5,2,6,6,5,5,4,7,9,3,5,5,2,3,6
RE,16,4,2,6,5,2,7,10,2,7,4,6,7,8,5,6
SA,28,7,2,8,6,7,6,8,5,8,9,4,5,6,10,5
BR,36,13,5,12,6,7,9,9,14,8,10,8,11,11,11,20
BH,16,7,1,9,6,4,3,0,6,3,6,2,3,5,5,4
CI,6,6,1,5,8,5,9,4,4,3,7,2,5,3,6,4


---
## 2. Alocação de passageiros (TAB_AP)

*A implementar.* Entrada: `Plano` decodificado. Saída: `Voo.ocupacao` preenchido + TAB_AP,
priorizando voo direto → 1 escala → 2 escalas, com capacidade de 200 e escala mínima de 60 min.

---
## 3. Função de avaliação (fitness)

*A implementar.*

---
## 4. Algoritmo genético e simulações

*A implementar.*